# Occlusion Detection in Autonomous Vehicles — nuScenes EDA

This notebook explores the `nuScenes v1.0-mini` dataset located in `data/` as a starting point for a research project on **occlusion detection** in autonomous driving perception.

The dataset ships as:
- `data/v1.0-mini/*.json` — relational metadata tables (scenes, samples, sensors, 3D annotations, ...)
- `data/samples/<CHANNEL>/` — annotated keyframe images/point clouds (one per sample, per sensor)
- `data/sweeps/<CHANNEL>/` — intermediate, unannotated frames between keyframes
- `data/maps/` — birds-eye-view map rasters for each location

Of particular interest for occlusion research: every 3D object annotation carries a **`visibility` label** (0-40%, 40-60%, 60-80%, 80-100% visible), which is the closest thing nuScenes provides to ground-truth occlusion. We'll use it throughout.

In [ ]:
import json
import os
import random
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100
RNG_SEED = 42
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

DATA_ROOT = "data"
META_ROOT = os.path.join(DATA_ROOT, "v1.0-mini")

print("Data root exists:", os.path.isdir(DATA_ROOT))
print("Metadata files:", sorted(os.listdir(META_ROOT)))

## 1. Load metadata tables

nuScenes stores its metadata as a set of linked JSON tables (similar to a relational database). We load every table into a `pandas.DataFrame`, indexed by `token` where present.

In [ ]:
TABLES = [
    "attribute", "calibrated_sensor", "category", "ego_pose", "instance",
    "log", "map", "sample", "sample_annotation", "sample_data",
    "scene", "sensor", "visibility",
]

def load_table(name: str) -> pd.DataFrame:
    path = os.path.join(META_ROOT, f"{name}.json")
    with open(path, encoding="utf-8") as fh:
        records = json.load(fh)
    return pd.DataFrame(records)

tables = {name: load_table(name) for name in TABLES}

overview = pd.DataFrame(
    {"rows": [len(df) for df in tables.values()]}, index=tables.keys()
).sort_values("rows", ascending=False)
overview

**Reading the table:**
- `sample` = one synchronized keyframe (404 total) grouped into `scene`s (10 scenes)
- `sample_data` = every individual sensor reading (keyframes *and* sweeps, 31k+ across 6 cameras + lidar + 5 radars)
- `sample_annotation` = every 3D bounding box for every object at every keyframe (18.5k)
- `instance` = a unique physical object tracked across a scene (911 objects)
- `visibility` = the 4 occlusion buckets we'll use as our occlusion proxy

## 2. Scenes, logs & locations

In [ ]:
scene = tables["scene"].merge(
    tables["log"], left_on="log_token", right_on="token", suffixes=("", "_log")
)
scene_display = scene[["name", "nbr_samples", "location", "vehicle", "date_captured", "description"]]
scene_display

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

scene.sort_values("nbr_samples").plot.barh(
    x="name", y="nbr_samples", ax=axes[0], legend=False, color="steelblue"
)
axes[0].set_title("Keyframes (samples) per scene")
axes[0].set_xlabel("# samples")
axes[0].set_ylabel("")

scene["location"].value_counts().plot.bar(ax=axes[1], color="indianred")
axes[1].set_title("Scenes per location")
axes[1].set_ylabel("# scenes")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

The mini split covers 10 short scenes (~20s each) split across **Boston Seaport** and two **Singapore** locations, captured on different dates — i.e. different lighting, weather and traffic conditions, which matters for an occlusion study.

## 3. Sensor rig & file inventory

Each vehicle carries 6 cameras (360° coverage), 1 lidar and 5 radars. `samples/` holds only the annotated keyframes; `sweeps/` holds the much larger set of intermediate frames.

In [ ]:
def count_files(split: str) -> pd.DataFrame:
    root = os.path.join(DATA_ROOT, split)
    rows = []
    for channel in sorted(os.listdir(root)):
        channel_dir = os.path.join(root, channel)
        if os.path.isdir(channel_dir):
            rows.append({"split": split, "channel": channel, "n_files": len(os.listdir(channel_dir))})
    return pd.DataFrame(rows)

file_counts = pd.concat([count_files("samples"), count_files("sweeps")], ignore_index=True)
file_counts_pivot = file_counts.pivot(index="channel", columns="split", values="n_files")
file_counts_pivot["modality"] = np.select(
    [file_counts_pivot.index.str.startswith("CAM"),
     file_counts_pivot.index.str.startswith("RADAR"),
     file_counts_pivot.index.str.startswith("LIDAR")],
    ["camera", "radar", "lidar"],
    default="other",
)
file_counts_pivot.sort_values("modality")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
file_counts_pivot[["samples", "sweeps"]].sort_values("samples").plot.barh(
    stacked=True, ax=ax, color=["steelblue", "lightgray"]
)
ax.set_xlabel("# files")
ax.set_title("Files per sensor channel (samples = annotated keyframes, sweeps = raw frames)")
plt.tight_layout()
plt.show()

Every channel has exactly **404** keyframes (= number of samples), confirming all 6 cameras (plus lidar/radar) are perfectly synchronized at the keyframe rate. `sweeps/` has ~5-9x more frames per channel since cameras/radars run faster than the 2 Hz keyframe rate and lidar sweeps continuously.

## 4. Image properties

We focus on the 6 camera channels since occlusion detection is primarily an image-understanding problem. `sample_data` already records `height`/`width` for every camera frame, so we can check resolution consistency without opening every file.

In [ ]:
sample_data = tables["sample_data"].copy()
cam_data = sample_data[sample_data["filename"].str.contains("CAM")].copy()
cam_data["channel"] = cam_data["filename"].str.extract(r"__(CAM_[A-Z_]+)__")

res_summary = cam_data.groupby("channel").agg(
    width=("width", "unique"), height=("height", "unique"), n_frames=("token", "count")
)
res_summary

All cameras deliver a single fixed resolution — no letterboxing/cropping to worry about when building an occlusion dataset.

In [ ]:
# File-size distribution per camera channel (JPEG compression artifacts can affect small/occluded objects)
N_PER_CHANNEL = 60  # keep the notebook fast; raise this for a deeper pass

cam_channels = sorted(cam_data["channel"].dropna().unique())
size_rows = []
sampled_paths = {}
for ch in cam_channels:
    subset = cam_data[cam_data["channel"] == ch]
    sampled = subset.sample(n=min(N_PER_CHANNEL, len(subset)), random_state=RNG_SEED)
    sampled_paths[ch] = sampled["filename"].tolist()
    for fname in sampled["filename"]:
        fsize = os.path.getsize(os.path.join(DATA_ROOT, fname))
        size_rows.append({"channel": ch, "size_kb": fsize / 1024})

size_df = pd.DataFrame(size_rows)
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=size_df, x="channel", y="size_kb", hue="channel", legend=False, ax=ax, palette="crest")
ax.set_title(f"JPEG file size distribution per camera (n={N_PER_CHANNEL}/channel sample)")
ax.set_ylabel("File size (KB)")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

## 5. Surround-view sample

One sample = one synchronized snapshot from every sensor. Let's visualize all 6 cameras for a single sample to see the full 360° coverage a perception stack has to reason about.

In [ ]:
def load_key_sample_images(sample_token: str) -> dict:
    rows = sample_data[
        (sample_data["sample_token"] == sample_token)
        & (sample_data["is_key_frame"] == True)
        & (sample_data["filename"].str.contains("CAM"))
    ]
    images = {}
    for _, row in rows.iterrows():
        channel = row["filename"].split("__")[1]
        images[channel] = Image.open(os.path.join(DATA_ROOT, row["filename"]))
    return images

demo_sample_token = tables["sample"].iloc[len(tables['sample']) // 2]["token"]
images = load_key_sample_images(demo_sample_token)

layout = ["CAM_FRONT_LEFT", "CAM_FRONT", "CAM_FRONT_RIGHT",
          "CAM_BACK_LEFT", "CAM_BACK", "CAM_BACK_RIGHT"]
fig, axes = plt.subplots(2, 3, figsize=(15, 6))
for ax, ch in zip(axes.flat, layout):
    if ch in images:
        ax.imshow(images[ch])
    ax.set_title(ch, fontsize=10)
    ax.axis("off")
fig.suptitle(f"6-camera surround view — sample {demo_sample_token[:8]}")
plt.tight_layout()
plt.show()

## 6. Pixel-level statistics: brightness & contrast per camera

Different mounting positions face different lighting (e.g. front cameras catch sun glare, side cameras catch shadows). Brightness/contrast affects how easy it is to spot partially-occluded objects, so it's worth profiling per channel.

In [ ]:
def image_stats(path: str) -> dict:
    with Image.open(path) as img:
        arr = np.asarray(img.convert("RGB"), dtype=np.float32)
    gray = arr.mean(axis=2)
    return {
        "brightness": gray.mean(),
        "contrast": gray.std(),
        "r_mean": arr[..., 0].mean(),
        "g_mean": arr[..., 1].mean(),
        "b_mean": arr[..., 2].mean(),
    }

stat_rows = []
for ch, filenames in sampled_paths.items():
    for fname in filenames:
        stats = image_stats(os.path.join(DATA_ROOT, fname))
        stats["channel"] = ch
        stat_rows.append(stats)

pixel_stats = pd.DataFrame(stat_rows)
pixel_stats.groupby("channel")[["brightness", "contrast"]].agg(["mean", "std"]).round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(data=pixel_stats, x="channel", y="brightness", hue="channel", legend=False, ax=axes[0], palette="flare")
axes[0].set_title("Brightness by camera")
axes[0].tick_params(axis="x", rotation=25)

sns.boxplot(data=pixel_stats, x="channel", y="contrast", hue="channel", legend=False, ax=axes[1], palette="crest")
axes[1].set_title("Contrast (grayscale std) by camera")
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

## 7. Time of day / lighting conditions

`sample.timestamp` is a Unix microsecond timestamp. We can't recover true local time without each log's UTC offset, but the *relative* hour distribution still tells us whether the mini split includes both daytime and low-light driving — a key confound for occlusion detection (occluded objects are harder to see in the dark).

In [ ]:
sample_df = tables["sample"].copy()
sample_df["datetime_utc"] = pd.to_datetime(sample_df["timestamp"], unit="us", utc=True)
sample_df["hour_utc"] = sample_df["datetime_utc"].dt.hour
sample_df = sample_df.merge(
    scene[["token", "name", "location"]], left_on="scene_token", right_on="token", suffixes=("", "_scene")
)

fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(data=sample_df, x="hour_utc", hue="location", multiple="stack", bins=24, ax=ax)
ax.set_title("Keyframe timestamps by UTC hour (proxy for lighting conditions)")
ax.set_xlabel("Hour (UTC)")
plt.tight_layout()
plt.show()

## 8. Object category distribution

Every annotation belongs to an `instance`, which belongs to a fine-grained `category` (e.g. `vehicle.car`, `human.pedestrian.adult`). We roll these up into coarse groups for readability.

In [ ]:
ann = tables["sample_annotation"].copy()
instance = tables["instance"][["token", "category_token"]].rename(columns={"token": "instance_token"})
category = tables["category"][["token", "name"]].rename(columns={"token": "category_token", "name": "category"})

ann = ann.merge(instance, on="instance_token").merge(category, on="category_token")
ann["category_coarse"] = ann["category"].str.split(".").str[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ann["category"].value_counts().plot.barh(ax=axes[0], color="teal")
axes[0].invert_yaxis()
axes[0].set_title("Annotations per fine-grained category")

ann["category_coarse"].value_counts().plot.bar(ax=axes[1], color="darkorange")
axes[1].set_title("Annotations per coarse category group")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## 9. Occlusion / visibility distribution — the core signal

`sample_annotation.visibility_token` links to the 4-level visibility scale. This is nuScenes' native occlusion label: the fraction of the object's projected footprint that is actually visible across all 6 cameras.

In [ ]:
visibility = tables["visibility"].rename(columns={"token": "visibility_token", "level": "visibility_level"})
ann = ann.merge(visibility[["visibility_token", "visibility_level", "description"]], on="visibility_token")

vis_order = ["v0-40", "v40-60", "v60-80", "v80-100"]
ann["visibility_level"] = pd.Categorical(ann["visibility_level"], categories=vis_order, ordered=True)

vis_counts = ann["visibility_level"].value_counts().reindex(vis_order)
vis_pct = (vis_counts / vis_counts.sum() * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
vis_counts.plot.bar(ax=axes[0], color=["#b30000", "#e34a33", "#fdbb84", "#2b8cbe"])
axes[0].set_title("Annotations per visibility bucket")
axes[0].set_ylabel("# annotations")
for i, v in enumerate(vis_counts):
    axes[0].text(i, v, f"{vis_pct.iloc[i]}%", ha="center", va="bottom")

axes[1].pie(vis_counts, labels=vis_order, autopct="%1.1f%%",
            colors=["#b30000", "#e34a33", "#fdbb84", "#2b8cbe"])
axes[1].set_title("Visibility share")
plt.tight_layout()
plt.show()

print(f"{(vis_counts[['v0-40','v40-60','v60-80']].sum() / vis_counts.sum() * 100):.1f}% "
      "of all annotations are at least partially occluded (< 80% visible).")

### Which categories get occluded the most?

In [ ]:
top_categories = ann["category"].value_counts().head(10).index
cross = pd.crosstab(ann[ann["category"].isin(top_categories)]["category"],
                     ann[ann["category"].isin(top_categories)]["visibility_level"],
                     normalize="index")[vis_order] * 100
cross = cross.loc[cross["v80-100"].sort_values().index]  # sort so most-occluded categories are on top

fig, ax = plt.subplots(figsize=(9, 6))
cross.plot.barh(stacked=True, ax=ax, color=["#b30000", "#e34a33", "#fdbb84", "#2b8cbe"])
ax.set_xlabel("% of annotations")
ax.set_title("Visibility-level breakdown for the 10 most common categories")
ax.legend(title="visibility", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### Lidar point count as a physical occlusion signal

`num_lidar_pts` counts how many lidar returns fall inside each 3D box. Heavily occluded or distant objects intercept fewer beams, so this is an independent, physics-based corroboration of the visibility label.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(
    data=ann, x="visibility_level", y="num_lidar_pts", hue="visibility_level", legend=False,
    order=vis_order, ax=ax, showfliers=False, palette=["#b30000", "#e34a33", "#fdbb84", "#2b8cbe"],
)
ax.set_yscale("symlog")
ax.set_title("Lidar points inside box vs. visibility level (outliers hidden, log scale)")
plt.tight_layout()
plt.show()

### Distance to ego vehicle vs. visibility

Objects further from the ego vehicle tend to be smaller in the image and more prone to being fully occluded by nearer objects. We approximate ego position at each sample via the `LIDAR_TOP` keyframe's `ego_pose`, then compute the planar distance to every annotated object.

In [ ]:
lidar_key = sample_data[
    (sample_data["filename"].str.contains("LIDAR_TOP")) & (sample_data["is_key_frame"] == True)
][["sample_token", "ego_pose_token"]]

ego_pose = tables["ego_pose"][["token", "translation"]].rename(
    columns={"token": "ego_pose_token", "translation": "ego_translation"}
)
sample_ego = lidar_key.merge(ego_pose, on="ego_pose_token")

ann_dist = ann.merge(sample_ego[["sample_token", "ego_translation"]], on="sample_token")
ego_xy = np.stack(ann_dist["ego_translation"].values)[:, :2]
obj_xy = np.stack(ann_dist["translation"].values)[:, :2]
ann_dist["distance_m"] = np.linalg.norm(obj_xy - ego_xy, axis=1)

fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(
    data=ann_dist, x="visibility_level", y="distance_m", hue="visibility_level", legend=False,
    order=vis_order, ax=ax, showfliers=False, palette=["#b30000", "#e34a33", "#fdbb84", "#2b8cbe"],
)
ax.set_ylabel("Distance to ego (m)")
ax.set_title("Object distance vs. visibility level")
plt.tight_layout()
plt.show()

## 10. Key findings

- The mini split has **10 scenes / 404 keyframes**, evenly covered by all 6 cameras + lidar + 5 radars (404 files each in `samples/`), plus a much larger `sweeps/` pool of unannotated frames.
- Two locations (**Boston Seaport**, **Singapore**) across several dates give some diversity in road layout and lighting.
- All camera frames share one fixed resolution per channel — no resizing artifacts to worry about.
- Brightness/contrast differ measurably across the 6 camera mounts, and keyframe timestamps span a wide hour range, so exposure/lighting is a real variable to control for.
- **~40% of all 3D annotations are less than 80% visible**, confirming occlusion is common rather than a rare edge case in this dataset.
- Occlusion rates vary strongly by category — some classes are systematically more occluded than others.
- `num_lidar_pts` and `distance_m` both correlate with the visibility label, giving two independent, physically-grounded ways to sanity-check or augment occlusion annotations.

## 11. Suggested next steps

1. Use `nuscenes-devkit` to project 3D boxes into each camera image (needs `camera_intrinsic` + `calibrated_sensor` + `ego_pose`) to get precise 2D occlusion masks/crops per object, rather than relying on the coarse 4-bucket label.
2. Build a per-object image-patch dataset labeled with `visibility_level` for training/evaluating an occlusion classifier or an amodal-completion model.
3. Investigate occluder-occludee relationships (which categories occlude which) using overlapping 2D projections within the same camera frame.
4. Extend this EDA to the full nuScenes trainval split once available, since the mini split (10 scenes) is only useful for prototyping, not for training.